# Train YOLOv8 phát hiện bong bóng thoại — YingHua

Notebook chạy được trên **Kaggle** (khuyên dùng) và **Google Colab**, tự nhận biết đang ở đâu.

Hai chế độ (đặt `MODE` ở Bước 2):
- **`seg` (mặc định)** — có **viền bong bóng**. Dataset chỉ có khung chữ nhật nên notebook dùng
  **SAM 2.1** sinh viền từ từng khung gốc, lọc viền hỏng, rồi train `yolov8m-seg`
  tiếp từ model cũ của dự án (`model.pt`, cũng là yolov8m-seg một lớp).
- **`detect`** — chỉ khung chữ nhật, train `yolov8s`, nhanh hơn.

| Bước | Việc | Kaggle (T4) | Colab (T4) |
|---|---|---|---|
| 1–2 | Cài thư viện, nạp dữ liệu | 2–5 phút | 3–6 phút |
| 3 | Sinh viền bằng SAM (chỉ `seg`) | 15–30 phút | 15–30 phút |
| 4 | Train | ~4–7 giờ (`seg`, 1280px, 100 epoch) | ~2,5–4 giờ (`seg`, 1024px, 60 epoch) |
| 5 | Đánh giá tập test + so với model cũ | 2 phút | 2 phút |
| 6–7 | Thử ảnh, xuất `best.pt` | tùy | tùy |

**Chạy thử trước:** đặt `QUICK_TEST = True` ở Bước 2 → 2 epoch trên 5% dữ liệu (~10 phút)
để chắc mọi ô chạy được, rồi đặt lại `False` để train thật.

**Giấy phép dataset:** CC BY-NC 4.0 — chỉ dùng phi thương mại; ghi nguồn
*PSImera/manga_bubbles_detect* (Roboflow manga-6puie + DLS_Manga_Translator) trong báo cáo.

## Bước 0 — Chuẩn bị (làm một lần)

**Trên máy:** mở PowerShell tại `D:\comic-dataset\manga_bubbles_detect`, nén ảnh + nhãn (~750 MB):
```
tar -a -cf bubbles_dataset.zip images labels
```

### Nếu dùng Kaggle (khuyên dùng — 30 giờ GPU/tuần, chạy nền được)
1. Đăng ký [kaggle.com](https://www.kaggle.com), vào **Settings → Phone verification** (bắt buộc để bật GPU và Internet).
2. **Datasets → New Dataset**, tải lên `bubbles_dataset.zip` **và** file model cũ
   `module xử lí ảnh/mudule xu ly anh/models bong bóng/model.pt`, để chế độ **Private**, đặt tên ví dụ `yinghua-bubbles`.
   (Kaggle tự giải nén zip.)
3. **Code → New Notebook → File → Import Notebook**, chọn file `.ipynb` này.
4. Thanh bên phải: **Add Input** → chọn dataset `yinghua-bubbles`; **Session options → Accelerator: GPU T4 x2**; **Internet: On**.
5. Chạy thử từng ô với `QUICK_TEST = True`. Ổn rồi thì đặt `False` và bấm **Save Version → Save & Run All (Commit)** —
   có thể tắt máy, Kaggle tự chạy hết; kết quả nằm ở tab **Output** của phiên bản đó.
6. Muốn tự thử ảnh của mình ở Bước 6 trên Kaggle: thêm một dataset chứa thư mục `test_images/` rồi Add Input.

### Nếu dùng Colab
1. Tải `bubbles_dataset.zip` và `model.pt` lên Google Drive vào thư mục **`MyDrive/yinghua_yolo/`**.
2. Mở notebook trên Colab, **Runtime → Change runtime type → T4 GPU**, chạy lần lượt từng ô.
   Phải giữ tab mở; Colab ngắt thì chạy lại từ đầu — notebook tự **train tiếp** từ checkpoint trên Drive.

## Bước 1 — Nhận biết môi trường, cài thư viện

In [ ]:
import os, sys
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (cho kagglehub), nên nhận Kaggle qua biến môi trường riêng của Kaggle.
ON_KAGGLE = not ON_COLAB and 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print('Môi trường:', 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Máy local')

!nvidia-smi --query-gpu=name,memory.total --format=csv
# Ghim đúng phiên bản đang dùng trong module xử lý ảnh của dự án để best.pt nạp được ở máy local.
%pip install -q ultralytics==8.4.138

## Bước 2 — Cấu hình

In [ ]:
MODE       = 'seg'    # 'seg': có viền bong bóng (khuyên dùng) · 'detect': chỉ khung chữ nhật
QUICK_TEST = False    # True: 2 epoch trên 5% dữ liệu để kiểm tra notebook trước khi train thật

PATIENCE   = 20       # dừng sớm nếu 20 epoch không cải thiện
BATCH      = -1       # -1: tự chọn theo bộ nhớ GPU (~60%); báo CUDA OOM thì đặt số cụ thể (vd. 4)
SAM_MODEL  = 'sam2.1_b.pt'
OLD_MODEL_NAME = 'model.pt'   # model cũ của dự án (yolov8m-seg) để train tiếp

if ON_KAGGLE:
    IMGSZ, EPOCHS = 1280, 100            # khớp imgsz=1280 mà src/pipeline.py dùng khi dự đoán
    OUT_DIR = Path('/kaggle/working')    # mọi thứ ở đây thành tab Output của phiên bản
    WORK = Path('/tmp/yinghua')          # dữ liệu tạm, không đưa vào Output
    SEARCH_ROOTS = [Path('/kaggle/input')]
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    IMGSZ, EPOCHS = 1024, 60             # Colab miễn phí hay bị ngắt nên train ngắn hơn
    OUT_DIR = Path('/content/drive/MyDrive/yinghua_yolo')   # lưu trên Drive để không mất khi bị ngắt
    WORK = Path('/content/yinghua')
    SEARCH_ROOTS = [WORK / 'raw', OUT_DIR]
else:
    IMGSZ, EPOCHS = 640, 30
    OUT_DIR = Path.cwd() / 'yinghua_out'
    WORK = Path.cwd() / 'yinghua_work'
    SEARCH_ROOTS = [Path.cwd()]

if QUICK_TEST:
    EPOCHS = 2
FRACTION = 0.05 if QUICK_TEST else 1.0
BASE_MODEL = {'seg': 'yolov8m-seg.pt', 'detect': 'yolov8s.pt'}[MODE]
RUN_NAME = f"bubbles_{MODE}_{IMGSZ}" + ('_quick' if QUICK_TEST else '')
RUNS_DIR = OUT_DIR / 'runs'
SPLITS = ('train', 'val', 'test')
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f'MODE={MODE} IMGSZ={IMGSZ} EPOCHS={EPOCHS} FRACTION={FRACTION} → {RUNS_DIR / RUN_NAME}')

## Bước 2b — Nạp dữ liệu

In [ ]:
import shutil, zipfile

def find_dataset_root(roots):
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('images/train'):
            if (p.parent.parent / 'labels' / 'train').is_dir():
                return p.parent.parent
    return None

def find_file(roots, name):
    for root in roots:
        if root.exists():
            hit = next(root.rglob(name), None)
            if hit:
                return hit
    return None

SEARCH_ROOTS = [WORK / 'raw', *SEARCH_ROOTS]
if find_dataset_root(SEARCH_ROOTS) is None:
    # Colab luôn phải giải nén từ Drive; Kaggle đôi khi giữ nguyên file zip thay vì tự giải nén.
    zip_path = find_file(SEARCH_ROOTS, '*.zip')
    if zip_path is None and ON_COLAB:
        # Chỉ quét 2 cấp của Drive để chỉ chỗ file đang nằm; quét hết Drive có thể rất lâu.
        my_drive = Path('/content/drive/MyDrive')
        found = [p for pattern in ('*.zip', '*/*.zip') for p in my_drive.glob(pattern)]
        print('Notebook tìm file zip trong:', OUT_DIR)
        print('Các file .zip đang có trên Drive:', *found or ['(không có)'], sep='\n  ')
    assert zip_path is not None, (f'Không thấy dataset. Đặt bubbles_dataset.zip vào {OUT_DIR} '
                                  '(hoặc sửa OUT_DIR ở Bước 2) rồi chạy lại ô này.')
    print('Giải nén', zip_path)
    local_zip = WORK / zip_path.name
    shutil.copy(zip_path, local_zip)          # giải nén thẳng từ Drive rất chậm
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(WORK / 'raw')
    local_zip.unlink()

RAW = find_dataset_root(SEARCH_ROOTS)
assert RAW is not None, 'Không tìm thấy thư mục images/train + labels/train — xem lại Bước 0.'
OLD_MODEL = find_file(SEARCH_ROOTS, OLD_MODEL_NAME)
print('Dataset :', RAW)
print('Model cũ:', OLD_MODEL or 'không có (sẽ train từ ' + BASE_MODEL + ')')

def link_or_copy(src, dst):
    # YOLO tìm nhãn bằng cách đổi /images/ thành /labels/ trong đường dẫn ảnh, nên mỗi bộ nhãn cần
    # một thư mục gốc riêng. Không dùng symlink thư mục: ultralytics lần theo symlink về thư mục gốc
    # và đọc nhầm nhãn khung. Hard link từng file không tốn thêm dung lượng; khác ổ đĩa thì chép.
    if dst.is_symlink():
        dst.unlink()            # dọn symlink do bản notebook cũ tạo
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        target = dst / f.name
        if f.is_file() and not target.exists():
            try:
                os.link(f, target)
            except OSError:
                shutil.copy2(f, target)

def write_yaml(root):
    yaml_path = root / 'data.yaml'
    yaml_path.write_text(
        f"path: {root.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        "names:\n  0: speech_bubble\n",   # trùng CLASS_NAMES[0] trong src/pipeline.py
        encoding='utf-8')
    return yaml_path

DET_DIR, SEG_DIR = WORK / 'det', WORK / 'seg'
for split in SPLITS:
    link_or_copy(RAW / 'images' / split, DET_DIR / 'images' / split)
    link_or_copy(RAW / 'labels' / split, DET_DIR / 'labels' / split)
    link_or_copy(RAW / 'images' / split, SEG_DIR / 'images' / split)
    (SEG_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
DATA_YAML = {'detect': write_yaml(DET_DIR), 'seg': write_yaml(SEG_DIR)}

for split in SPLITS:
    n_img = len(list((RAW / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((RAW / 'labels' / split).glob('*.txt')))
    print(f'{split:5s}: {n_img:5d} ảnh, {n_lbl:5d} nhãn')

## Bước 2c — Xem thử nhãn gốc (khung)

In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

COLOR = (164, 48, 115)   # màu chủ đạo YingHua #a43073

def draw_labels(img_path, label_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lines = [l.split() for l in label_path.read_text().splitlines() if l.strip()] if label_path.exists() else []
    thick = max(2, w // 300)
    for parts in lines:
        vals = [float(v) for v in parts[1:]]
        if len(vals) == 4:     # khung: cx cy w h
            cx, cy, bw, bh = vals[0] * w, vals[1] * h, vals[2] * w, vals[3] * h
            cv2.rectangle(img, (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2)), COLOR, thick)
        else:                  # đa giác: x1 y1 x2 y2 ...
            pts = (np.array(vals).reshape(-1, 2) * [w, h]).astype(np.int32)
            cv2.polylines(img, [pts], True, COLOR, thick)
    return img, len(lines)

def show_label_samples(root, split='train', k=4, seed=0):
    # Chỉ chọn ảnh có nhãn (QUICK_TEST chỉ sinh viền cho một phần ảnh).
    paths = [p for p in sorted((root / 'images' / split).glob('*.jpg'))
             if (root / 'labels' / split / (p.stem + '.txt')).is_file()
             and (root / 'labels' / split / (p.stem + '.txt')).stat().st_size > 0]
    if not paths:
        print('Chưa có ảnh nào có nhãn trong', root / 'labels' / split)
        return
    picks = random.Random(seed).sample(paths, min(k, len(paths)))
    fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 8))
    for p, ax in zip(picks, np.atleast_1d(axes)):
        img, n = draw_labels(p, root / 'labels' / split / (p.stem + '.txt'))
        ax.imshow(img); ax.set_title(f'{p.name} — {n} bong bóng'); ax.axis('off')
    plt.show()

show_label_samples(DET_DIR)

## Bước 3 — Sinh viền bong bóng bằng SAM (chỉ khi `MODE = 'seg'`)

Với mỗi khung gốc, SAM 2.1 trả về mặt nạ đúng hình bong bóng. Mặt nạ bị **cắt trong khung gốc**
(bong bóng không thể vượt khung đã gán nhãn), rồi lọc:
- phủ < 35% khung, hoặc khung bao của viền < 60% khung gốc → coi là hỏng (thường gặp ở chữ nằm thẳng trên tranh)
  → **dùng chính khung chữ nhật làm đa giác** để không mất nhãn.

Kết quả lưu thành `seg_labels.zip` trong thư mục output; lần chạy sau (Colab) tự dùng lại, không chạy SAM nữa.

In [ ]:
from collections import Counter
from tqdm.auto import tqdm

MIN_FILL = 0.35   # diện tích viền / diện tích khung
MIN_SPAN = 0.60   # diện tích khung bao của viền / diện tích khung
SEG_CACHE = OUT_DIR / 'seg_labels.zip'

def read_boxes(label_path, w, h):
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cx, cy, bw, bh = (float(v) for v in parts[1:])
            x1, y1 = max(0, int((cx - bw / 2) * w)), max(0, int((cy - bh / 2) * h))
            x2, y2 = min(w, int((cx + bw / 2) * w)), min(h, int((cy + bh / 2) * h))
            if x2 - x1 > 1 and y2 - y1 > 1:
                boxes.append([x1, y1, x2, y2])
    return boxes

def mask_to_polygon(mask, box):
    x1, y1, x2, y2 = box
    clipped = np.zeros_like(mask)
    clipped[y1:y2, x1:x2] = mask[y1:y2, x1:x2]
    contours, _ = cv2.findContours(clipped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    box_area = (x2 - x1) * (y2 - y1)
    _, _, cw, ch = cv2.boundingRect(c)
    if cv2.contourArea(c) / box_area < MIN_FILL or cw * ch / box_area < MIN_SPAN:
        return None
    c = cv2.approxPolyDP(c, 0.002 * cv2.arcLength(c, True), True).reshape(-1, 2)
    return c if len(c) >= 3 else None

def box_polygon(box):
    x1, y1, x2, y2 = box
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])

def to_yolo_line(poly, w, h):
    return '0 ' + ' '.join(f'{x / w:.6f} {y / h:.6f}' for x, y in poly)

def generate_seg_labels():
    from ultralytics import SAM
    sam = SAM(SAM_MODEL)
    stats = Counter()
    for split in SPLITS:
        paths = sorted((RAW / 'images' / split).glob('*.jpg'))
        if QUICK_TEST:
            paths = paths[:max(8, int(len(paths) * FRACTION))]
        for img_path in tqdm(paths, desc=f'SAM {split}'):
            h, w = cv2.imread(str(img_path)).shape[:2]
            boxes = read_boxes(RAW / 'labels' / split / (img_path.stem + '.txt'), w, h)
            lines = []
            if boxes:
                r = sam(str(img_path), bboxes=boxes, verbose=False)[0]
                masks = r.masks.data.cpu().numpy() if r.masks is not None else []
                for i, box in enumerate(boxes):
                    poly = None
                    if i < len(masks):
                        m = (masks[i] > 0.5).astype(np.uint8)
                        if m.shape != (h, w):
                            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
                        poly = mask_to_polygon(m, box)
                    stats['viền SAM' if poly is not None else 'dùng khung'] += 1
                    lines.append(to_yolo_line(poly if poly is not None else box_polygon(box), w, h))
            (SEG_DIR / 'labels' / split / (img_path.stem + '.txt')).write_text('\n'.join(lines))
    return stats

if MODE == 'seg':
    if SEG_CACHE.exists() and not QUICK_TEST:
        with zipfile.ZipFile(SEG_CACHE) as zf:
            zf.extractall(SEG_DIR)
        print('Dùng lại nhãn viền đã sinh:', SEG_CACHE)
    else:
        stats = generate_seg_labels()
        total = sum(stats.values())
        for k, v in stats.items():
            print(f'{k:10s}: {v:6d} ({v / total:.1%})')
        if not QUICK_TEST:
            with zipfile.ZipFile(SEG_CACHE, 'w', zipfile.ZIP_DEFLATED) as zf:
                for f in (SEG_DIR / 'labels').rglob('*.txt'):
                    zf.write(f, f.relative_to(SEG_DIR).as_posix())
            print('Đã lưu', SEG_CACHE)
    show_label_samples(SEG_DIR)
    show_label_samples(SEG_DIR, split='val', seed=1)

## Bước 4 — Train

- `seg`: train tiếp từ model cũ của dự án nếu tìm thấy, không thì từ `yolov8m-seg.pt` (học sẵn trên COCO).
- Có `last.pt` dở dang của lần trước (Colab bị ngắt) → **train tiếp**; đã train xong → bỏ qua.

In [ ]:
import torch
from ultralytics import YOLO

run_dir = RUNS_DIR / RUN_NAME
last_ckpt = run_dir / 'weights' / 'last.pt'
# Ultralytics ghi epoch = -1 vào last.pt khi đã train xong; resume lúc đó sẽ báo lỗi.
finished = last_ckpt.exists() and torch.load(last_ckpt, map_location='cpu', weights_only=False).get('epoch', -1) < 0

if finished:
    print('Lần chạy', RUN_NAME, 'đã train xong → bỏ qua. Muốn train lại thì đổi RUN_NAME ở Bước 2.')
elif last_ckpt.exists():
    print('Tìm thấy', last_ckpt, '→ train tiếp')
    YOLO(str(last_ckpt)).train(resume=True)
else:
    start = str(OLD_MODEL) if MODE == 'seg' and OLD_MODEL else BASE_MODEL
    print('Train từ', start)
    YOLO(start).train(
        data=str(DATA_YAML[MODE]),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=BATCH,
        fraction=FRACTION,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        save_period=5 if ON_COLAB else -1,   # checkpoint định kỳ chỉ cần khi có thể bị ngắt
        cos_lr=True,
        close_mosaic=10,
        fliplr=0.0,          # lật ngang làm ngược chiều đọc của truyện nên tắt
        workers=2,
        plots=True,
    )

In [ ]:
from IPython.display import Image, display

for name in ('results.png', 'MaskPR_curve.png', 'BoxPR_curve.png', 'PR_curve.png', 'val_batch0_pred.jpg'):
    f = run_dir / name
    if f.exists():
        print(name); display(Image(filename=str(f), width=1000))

## Bước 5 — Đánh giá trên tập test và so với model cũ

Tập **test** là ảnh model chưa từng thấy khi train. Chỉ số:
**mAP50** — bắt được bong bóng hay không (≥ 0,90 là dùng tốt) · **mAP50-95** — khung/viền có khít không.
Với `seg`, cột *mask* đo độ khít của viền.

In [ ]:
import pandas as pd

best_pt = run_dir / 'weights' / 'best.pt'

def evaluate(weights, label):
    r = YOLO(str(weights)).val(data=str(DATA_YAML[MODE]), split='test', imgsz=IMGSZ, batch=8,
                               project=str(RUNS_DIR), name=f'{RUN_NAME}_test_{label}', exist_ok=True,
                               plots=False, verbose=False)
    row = {'model': label, 'box P': r.box.mp, 'box R': r.box.mr, 'box mAP50': r.box.map50, 'box mAP50-95': r.box.map}
    if hasattr(r, 'seg'):
        row.update({'mask mAP50': r.seg.map50, 'mask mAP50-95': r.seg.map})
    return row

rows = [evaluate(best_pt, 'mới')]
if MODE == 'seg' and OLD_MODEL:
    rows.append(evaluate(OLD_MODEL, 'cũ'))
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
best = YOLO(str(best_pt))

def show_predictions(paths, conf=0.30, imgsz=1280):
    # conf=0.30 và imgsz=1280 giống cấu hình đang chạy trong src/pipeline.py
    results = best.predict([str(p) for p in paths], conf=conf, imgsz=imgsz, verbose=False)
    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 9))
    for r, ax in zip(results, np.atleast_1d(axes)):
        ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(f'{Path(r.path).name} — {len(r.boxes)} bong bóng'); ax.axis('off')
    plt.show()

show_predictions(random.Random(2).sample(sorted((RAW / 'images' / 'test').glob('*.jpg')), 4))

## Bước 6 — Thử với ảnh của bạn

- **Colab:** chạy ô dưới, bấm **Choose Files**, chọn trang truyện (jpg/png/webp).
- **Kaggle:** Add Input một dataset có thư mục `test_images/`; ô dưới tự tìm.

Nên thử cả trang **manhua chữ Hán** (màu, khổ dài): dataset chủ yếu là manga Nhật đen trắng, nếu manhua
sót nhiều bong bóng thì cần gán nhãn thêm ảnh manhua rồi train tiếp.
Mặc định `conf=0.30` như pipeline; sót thì hạ (0,2), bắt nhầm thì nâng (0,5).

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}
paths = []
if ON_COLAB:
    from google.colab import files
    upload_dir = WORK / 'uploads'; upload_dir.mkdir(exist_ok=True)
    for name, data in files.upload().items():
        p = upload_dir / Path(name).name
        p.write_bytes(data)
        paths.append(p)
else:
    test_dir = next((d for root in SEARCH_ROOTS if root.exists() for d in root.rglob('test_images') if d.is_dir()), None)
    if test_dir:
        paths = sorted(p for p in test_dir.iterdir() if p.suffix.lower() in IMG_EXT)
    else:
        print('Không có thư mục test_images/ — bỏ qua bước này.')
for i in range(0, len(paths), 4):
    show_predictions(paths[i:i + 4])

## Bước 7 — Xuất model

In [ ]:
export_pt = OUT_DIR / f'{RUN_NAME}_best.pt'
shutil.copy(best_pt, export_pt)
print('Đã lưu:', export_pt)
if ON_COLAB:
    from google.colab import files
    files.download(str(export_pt))
elif ON_KAGGLE:
    print('Tải về ở tab Output của phiên bản notebook này.')

## Đưa model vào dự án YingHua

1. Sao lưu model cũ: đổi tên
   `module xử lí ảnh/mudule xu ly anh/models bong bóng/model.pt` → `model_seg_old.pt`.
2. Chép file vừa tải về vào đúng chỗ đó, đặt tên `model.pt` — `src/pipeline.py` tự tìm file này, không cần sửa code.
3. Khởi động lại worker xử lý ảnh để nạp model mới, rồi xử lý lại vài trang để so với model cũ.

- Model `seg` có viền như model cũ nên bước xóa chữ dùng được viền ngay.
- Model `detect` chỉ có khung: pipeline vẫn chạy nhưng bước xóa chữ dùng khung chữ nhật — nên so sánh trước khi thay.